# Dataset Builder: EN / ZH / VI Multilingual QA Dataset

## 1. Introduction

This notebook builds three balanced JSONL files for human vs AI-generated text classification:

- `en_qa.jsonl` — 300 EN QA pairs (human: HC3 reddit_eli5, AI: Qwen2.5-1.5B-Instruct)
- `zh_qa.jsonl` — 300 ZH QA pairs (human: HC3-Chinese open_qa, AI: Qwen2.5-1.5B-Instruct)
- `vi_qa.jsonl` — 300 VI QA pairs (human: crawled VN Reddit, AI: Qwen2.5-1.5B-Instruct)

**Note on source selection:**
- HC3-EN uses `reddit_eli5` as the source (community-written, informal, diverse).
- HC3-ZH has no Reddit equivalent; `open_qa` is used as the closest general-domain human source.
- VI data was crawled from Vietnamese Q&A communities.

**Output schema per file:**
```json
{
  "question": "...",
  "human_answers": ["..."],
  "ai_answers": ["..."],
  "index": "...",
  "source": "..."
}

## 2. Installs

In [1]:
%pip install -q datasets transformers accelerate

Note: you may need to restart the kernel to use updated packages.


## 3. Imports and Config

In [2]:
import json
import random
from pathlib import Path
from datasets import load_dataset

SEED = 42
N = 300
random.seed(SEED)

CKPT_SUFFIX = ".ckpt.jsonl"
LANG_NAMES = {"EN": "English", "ZH": "Chinese", "VI": "Vietnamese"}

def make_prompt(question: str, language: str) -> str:
    lang = LANG_NAMES.get(language.upper(), language)
    return f'Answer this question in {lang} in 100-300 words: "{question}"'

## 4. Load and filter HC3-EN and HC3-ZH

We load both HC3 datasets, filter to the chosen source, and sample N=300 prompts.

- EN: filter `source == "reddit_eli5"` — informal community answers, most comparable to VI crawled data.
- ZH: filter `source == "open_qa"` — HC3-ZH has no Reddit subset; open_qa is the broadest general-domain source.

We keep only rows that have at least one valid human answer.

### 4.1 Load HC3-EN and HC3-ZH

In [3]:
# Load 
# HC3 English
# HC3 English (original)
hc3_en = load_dataset(
    "json",
    data_files="https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/all.jsonl",
)["train"]

# HC3 Chinese (official Chinese version)
hc3_zh = load_dataset(
    "json",
    data_files="https://huggingface.co/datasets/Hello-SimpleAI/HC3-Chinese/resolve/main/all.jsonl",
)["train"]

print(f"HC3-EN total rows: {len(hc3_en)}")
print(f"HC3-ZH total rows: {len(hc3_zh)}")
print(f"HC3-EN sources: {set(hc3_en['source'])}")
print(f"HC3-ZH sources: {set(hc3_zh['source'])}")

HC3-EN total rows: 24322
HC3-ZH total rows: 12853
HC3-EN sources: {'medicine', 'reddit_eli5', 'finance', 'open_qa', 'wiki_csai'}
HC3-ZH sources: {'law', 'medicine', 'baike', 'finance', 'psychology', 'nlpcc_dbqa', 'open_qa'}


### 4.2 Filter and Sample EN + ZH

In [4]:
def has_valid_human(row):
    answers = row.get("human_answers") or []
    return len(answers) > 0 and any(
        isinstance(a, str) and len(a.strip()) > 30 for a in answers
    )

def sample_prompts(dataset, source_filter, n, lang):
    filtered = [
        row for row in dataset
        if row.get("source", "") == source_filter and has_valid_human(row)
    ]
    print(f"[{lang}] after filter '{source_filter}': {len(filtered)} rows")
    if len(filtered) < n:
        print(f"  WARNING: using all {len(filtered)} available.")
    random.shuffle(filtered)
    return filtered[:n]

en_prompts = sample_prompts(hc3_en, "reddit_eli5", N, "EN")
zh_prompts = sample_prompts(hc3_zh, "open_qa", N, "ZH")
print(f"\nSelected: {len(en_prompts)} EN, {len(zh_prompts)} ZH")

[EN] after filter 'reddit_eli5': 17112 rows
[ZH] after filter 'open_qa': 3050 rows

Selected: 300 EN, 300 ZH


## 5. Load Vietnamese crawled data

Vietnamese human answers come from `vi_qa_crawled.jsonl`.
Expected schema: 
```json
{
  "question": "...",
  "human_answers": ["..."],
  "source": "...",
  "post_id": "..."
}
```

We standardize to the same intermediate format as HC3:
```json
{
  "question": "...",
  "human_answers": ["..."],
  "ai_answers": ["..."],
  "index": "...",
  "source": "..."
}
```

In [5]:
def load_vi_jsonl(path, n):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            q = (obj.get("question") or obj.get("title") or "").strip()
            raw = obj.get("human_answers") or obj.get("answer") or obj.get("body") or ""
            ans = (raw[0] if isinstance(raw, list) and raw else (raw if isinstance(raw, str) else "")).strip()
            if not q or not ans or len(ans) < 30:
                continue
            rows.append({
                "question": q,
                "human_answers": [ans],
                "index": obj.get("id") or obj.get("index") or obj.get("post_id"),
                "source": "vi_crawled",
            })
    print(f"[VI] valid rows: {len(rows)}")
    random.shuffle(rows)
    return rows[:n]

vi_prompts = load_vi_jsonl("vi_qa_crawled.jsonl", N)
print(f"Selected: {len(vi_prompts)} VI prompts")

[VI] valid rows: 301
Selected: 300 VI prompts


## 6. Generate AI Answers (Qwen2.5-1.5B-Instruct)

- **Prompt template:** `Answer this question in {language} in 100-300 words: "{question}"`
- **Settings:** 
    - max_new_tokens=512
    - temperature=0.7
    - top_p=0.9
    - do_sample=True
- **Strategy:** writes after each new generation; resumes correctly on restart

### 6.1 Load Qwen Model

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Model loaded.")

Loading Qwen/Qwen2.5-1.5B-Instruct ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded.


### 6.2 Define reuse functions

In [7]:
def generate_answer(prompt_text: str, max_new_tokens: int = 512) -> str:
    messages = [{"role": "user", "content": prompt_text}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def load_checkpoint(ckpt_path: str) -> dict:
    out = {}
    if not Path(ckpt_path).exists():
        return out
    with open(ckpt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            out[obj["question"]] = obj["ai_answer"]
    return out


# Generate AI answers and write output JSONL incrementally.
def process_language(prompts, lang, ckpt_path, output_path):
    ckpt = load_checkpoint(ckpt_path)
    if ckpt:
        print(f"[{lang}] Resumed {len(ckpt)} from checkpoint.")

    with open(output_path, "w", encoding="utf-8") as out_f:
        with open(ckpt_path, "a", encoding="utf-8") as ckpt_f:
            for i, row in enumerate(prompts):
                q = row["question"]
                if q in ckpt:
                    ai_answer = ckpt[q]
                else:
                    prompt_text = make_prompt(q, lang)
                    ai_answer = generate_answer(prompt_text)
                    ckpt[q] = ai_answer
                    ckpt_f.write(json.dumps({"question": q, "ai_answer": ai_answer}, ensure_ascii=False) + "\n")
                    ckpt_f.flush()

                human = row.get("human_answers") or []
                record = {
                    "question": q,
                    "human_answers": [human[0]] if human else [],
                    "ai_answers": [ai_answer],
                    "index": row.get("index"),
                    "source": row.get("source", "unknown"),
                }
                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")

                if (i + 1) % 50 == 0:
                    print(f"[{lang}] {i+1}/{len(prompts)}")

    print(f"[{lang}] Done -> {output_path}")

### 6.3 Run for en, zh, vi (one language at a time)

In [8]:
process_language(en_prompts, "EN", "en_qa" + CKPT_SUFFIX, "en_qa.jsonl")
process_language(zh_prompts, "ZH", "zh_qa" + CKPT_SUFFIX, "zh_qa.jsonl")
process_language(vi_prompts, "VI", "vi_qa" + CKPT_SUFFIX, "vi_qa.jsonl")

[EN] Resumed 300 from checkpoint.
[EN] 50/300
[EN] 100/300
[EN] 150/300
[EN] 200/300
[EN] 250/300
[EN] 300/300
[EN] Done -> en_qa.jsonl
[ZH] Resumed 300 from checkpoint.
[ZH] 50/300
[ZH] 100/300
[ZH] 150/300
[ZH] 200/300
[ZH] 250/300
[ZH] 300/300
[ZH] Done -> zh_qa.jsonl
[VI] Resumed 300 from checkpoint.
[VI] 50/300
[VI] 100/300
[VI] 150/300
[VI] 200/300
[VI] 250/300
[VI] 300/300
[VI] Done -> vi_qa.jsonl


## 7. Verify Output

In [9]:
for path, lang in [("en_qa.jsonl", "EN"), ("zh_qa.jsonl", "ZH"), ("vi_qa.jsonl", "VI")]:
    with open(path, "r", encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]
    print(f"\n[{lang}] {path}: {len(rows)} records")
    print(f"Keys: {list(rows[0].keys())}")
    print(f"Example question: {rows[0]['question'][:80]}...")
    print(f"Human[0][:60]: {rows[0]['human_answers'][0][:60]}...")
    print(f"AI[0][:60]: {rows[0]['ai_answers'][0][:60]}...")


[EN] en_qa.jsonl: 300 records
Keys: ['question', 'human_answers', 'ai_answers', 'index', 'source']
Example question: Since phone batteries must be so slim nowadays why ca n't phones just have two b...
Human[0][:60]: Because that would make the thin phones bigger ....
AI[0][:60]: Dear little one, you're asking some great questions about ho...

[ZH] zh_qa.jsonl: 300 records
Keys: ['question', 'human_answers', 'ai_answers', 'index', 'source']
Example question: 幽门杆菌检测为阴性了胃病也不一定已经治好了吗？最近做呼气试验检测HP为阴性（以前一直是阳性），但是我完全没有觉得我的浅表性胃炎有所好转，是不是就算清除了幽门杆菌...
Human[0][:60]: 谢邀。各个知友最近问了许多有关于慢性胃病的问题（在导致慢性胃炎的幽门螺旋杆菌会传染吗？ ），尽可能在这里一并回答。如题目...
AI[0][:60]: 关于您提到的“幽门杆菌检测为阴性”和“浅表性胃炎没有明显好转”的问题，我可以从以下几个方面进行解释：

首先，幽门杆菌感...

[VI] vi_qa.jsonl: 300 records
Keys: ['question', 'human_answers', 'ai_answers', 'index', 'source']
Example question: Cố bao nhiêu là đủ? 2015 - Năm vừa tốt nghiệp ĐH, nhà tôi vỡ nợ tầm 5 tỷ - bán c...
Human[0][:60]: tôi cũng 30. T rút ra đc 1 câu như này. cũng k phải nói toàn...
AI[0][:60]: Tôi h